In [1]:
from pathlib import Path
import sys
sys.path.append( str( Path("../../../.." ).resolve()) )

# <p align="center">  Tools to Perform Structural Alignment of Molecular Models </p>

In [2]:
# Load modules and data
import gemmi

from xaidar.data.molecModels import flatten_pdb

from xaidar.data.molecModels import get_pdb_stats, sele_pdb, sele_Lig, get_res_CoM, sele_closest_Chain
from xaidar.data.molecModels import flatten_pdb, sele_AA, createPDB, get_atom_coord

ref_prot = gemmi.read_structure(str(Path("../../../../tests/testdata/protein/A0926a.pdb").resolve()))
mobile_prot = gemmi.read_structure(str(Path("../../../../tests/testdata/protein/A7501a.pdb").resolve()))

prot_lig = sele_pdb( ref_prot, sele_Lig )
lig_CoM = gemmi.Position( *get_res_CoM( flatten_pdb(prot_lig, "residue") )[0])
# assign back to the original variables so changes persist
filtered_prots ={}
for name, prot in zip( ["ref_prot", "mobile_prot"], [ref_prot, mobile_prot] ):
    prot = sele_pdb(prot, sele_AA)
    prot = sele_pdb(prot, sele_closest_Chain, lig_CoM)
    filtered_prots[name] = prot


print( "Before:")
get_pdb_stats( ref_prot )
get_pdb_stats( mobile_prot )
print("\nAfter filtering:")
get_pdb_stats( filtered_prots["ref_prot"] )
get_pdb_stats( filtered_prots["mobile_prot"] )

ref_prot = filtered_prots["ref_prot"]
mobile_prot = filtered_prots["mobile_prot"]

Before:

####################
Number of models: 1
Number of chains in 1st Model: 4

Chain ID: A
	Number of Residues: 142
	Unique List of Non-A.A.: {'LIG', 'ZN'}
	Contains A.A.
Chain ID: B
	Number of Residues: 141
	Unique List of Non-A.A.: {'ZN'}
	Contains A.A.
Chain ID: C
	Number of Residues: 2
	Unique List of Non-A.A.: {'DMS'}
Chain ID: H
	Number of Residues: 9
	Unique List of Non-A.A.: {'HOH'}

####################
Number of models: 1
Number of chains in 1st Model: 5

Chain ID: A
	Number of Residues: 142
	Unique List of Non-A.A.: {'LIG', 'DMS'}
	Contains A.A.
Chain ID: B
	Number of Residues: 143
	Unique List of Non-A.A.: {'LIG', 'GOL', 'DMS'}
	Contains A.A.
Chain ID: C
	Number of Residues: 2
	Unique List of Non-A.A.: {'ZN'}
Chain ID: D
	Number of Residues: 18
	Unique List of Non-A.A.: {'HOH'}
Chain ID: E
	Number of Residues: 2
	Unique List of Non-A.A.: {'HOH'}

After filtering:

####################
Number of models: 1
Number of chains in 1st Model: 1

Chain ID: B
	Number of Residues

- ## Testing gemmi functions

In [ ]:
mobile_prot_align =mobile_prot.clone()
supresult_before = gemmi.calculate_current_rmsd( flatten_pdb(ref_prot, "chain")[0].whole(), 
                             flatten_pdb(mobile_prot_align, "chain" )[0].whole(),
                              ptype = flatten_pdb(ref_prot, "chain")[0].whole().check_polymer_type(),
                              sel = gemmi.SupSelect.All) # All, MainChain, CaP (only Cα atoms (for peptides) and P atoms (for nucleotides) )
rot_matrix = supresult_before.transform.mat # Rotation Matrix
trans_vect = supresult_before.transform.vec # Translation Vector
print("Rmsd: ", supresult_before.rmsd)
print("Rotation Matrix:\n", rot_matrix)
print("Translation Vector:\n", trans_vect)

print()

supresult_after = gemmi.calculate_superposition( flatten_pdb(ref_prot, "chain")[0].whole(),
                                             flatten_pdb(mobile_prot_align, "chain" )[0].whole(),
                                             flatten_pdb(ref_prot, "chain")[0].whole().check_polymer_type(),
                                             gemmi.SupSelect.All, )
                                            # trim_cutoff= 2.0)
rot_matrix = supresult_after.transform.mat # Rotation Matrix
trans_vect = supresult_after.transform.vec # Translation Vector
print("Rmsd: ", supresult_after.rmsd)
print("Rotation Matrix:\n", rot_matrix)
print("Translation Vector:\n", trans_vect)

print()

print( "Ref Prot Atom Pos: ", ref_prot[0]["B"][0][0].pos )
print( "Test Prot Atom Pos: ", mobile_prot[0]["A"][0][0].pos )
print( "Aligned Test Prot Atom Pos: ", mobile_prot_align[0]["A"][0][0].pos )
mobile_prot_align[0]["A"].whole().transform_pos_and_adp(supresult_after.transform)
print( "Aligned Test Prot Atom Pos: ", mobile_prot_align[0]["A"][0][0].pos )

Rmsd:  10.5925665200077
Rotation Matrix:
 <gemmi.Mat33 [1, 0, 0]
             [0, 1, 0]
             [0, 0, 1]>
Translation Vector:
 <gemmi.Vec3(0, 0, 0)>

Rmsd:  0.75135131336274
Rotation Matrix:
 <gemmi.Mat33 [0.670284, 0.596009, -0.442146]
             [-0.721815, 0.661976, -0.201918]
             [0.172345, 0.45449, 0.87392]>
Translation Vector:
 <gemmi.Vec3(7.0267, 17.6825, -6.98399)>

Ref Prot Atom Pos:  <gemmi.Position(17.43, 6.362, 25.238)>
Test Prot Atom Pos:  <gemmi.Position(20.808, 13.206, 25.917)>
Aligned Test Prot Atom Pos:  <gemmi.Position(20.808, 13.206, 25.917)>
Aligned Test Prot Atom Pos:  <gemmi.Position(17.3858, 6.17196, 25.2535)>


In [ ]:
from xaidar.data.molecModels import savePDB
for name, struct in zip( ["ref_prot.pdb", "mobile_prot.pdb", "mobile_prot_aligned.pdb"],
                         [ref_prot, mobile_prot, mobile_prot_align] ):
    savePDB( struct, name )

- ## align_protein() function

In [ ]:
# Function
def align_proteins( ref_prot: gemmi.Structure, mobile_prot: gemmi.Structure ) -> gemmi.Transform:
    supresult = gemmi.calculate_superposition( flatten_pdb(ref_prot, "chain")[0].whole(),
                                             flatten_pdb(mobile_prot, "chain" )[0].whole(),
                                              flatten_pdb(ref_prot, "chain")[0].whole().check_polymer_type(),
                                              gemmi.SupSelect.All, )
    flatten_pdb(mobile_prot, "chain" )[0].whole().transform_pos_and_adp(supresult.transform) # align prot
    return supresult.transform

In [ ]:
mobile_prot_align = mobile_prot.clone()
align_transform = align_proteins( ref_prot, mobile_prot_align )


print( "Ref Prot Atom Pos: ", ref_prot[0]["B"][0][0].pos )
print( "Test Prot Atom Pos: ", mobile_prot[0]["A"][0][0].pos )
print( "Aligned Test Prot Atom Pos: ", mobile_prot_align[0]["A"][0][0].pos )
mobile_prot_align[0]["A"].whole().transform_pos_and_adp(align_transform)
print( "Aligned Test Prot Atom Pos: ", mobile_prot_align[0]["A"][0][0].pos )

Ref Prot Atom Pos:  <gemmi.Position(17.43, 6.362, 25.238)>
Test Prot Atom Pos:  <gemmi.Position(20.808, 13.206, 25.917)>
Aligned Test Prot Atom Pos:  <gemmi.Position(17.3858, 6.17196, 25.2535)>
Aligned Test Prot Atom Pos:  <gemmi.Position(11.1929, 4.11977, 20.887)>


- ## Class structAlign()

In [4]:
class structAlign():
    def __init__( self, ref_prot:gemmi.Structure, mobile_prot:gemmi.Structure):
        self.ref_prot : gemmi.Structure = ref_prot
        self.mobile_prot: gemmi.Structure  = mobile_prot
        self.aligned_prot: gemmi.Structure | None  = None
        self.aligned_status: bool = False
        self.transform = None
        self.rmsd = None
        self.rot_matrix = None
        self.trans_vect = None

    def align( self, ref_atoms: str = "All") :
        """ 
        Align mobile_prot to ref_prot using gemmi superposition calculation 
        Currently, only takes one chain from each structure for alignment.
        Also, aligns all atoms in the chain (can be modified to select specific atoms).
        ref_atoms: str
            Atom selection for reference structure alignment. 
            Options: "All", "MainChain", "CaP"
        Returns:
        self: model_structAlign
            The instance with updated aligned_prot, transform, rmsd, rot_matrix, 
            trans_vect attributes.
        """
        self.aligned_prot = self.mobile_prot.clone()
        if ref_atoms not in ["All", "MainChain", "CaP"]:
            raise ValueError("ref_atoms must be one of 'All', 'MainChain', or 'CaP'.")

        supresult = gemmi.calculate_superposition( 
        flatten_pdb(self.ref_prot, "chain")[0].whole(),
        flatten_pdb(self.aligned_prot, "chain" )[0].whole(),
        flatten_pdb(self.aligned_prot, "chain")[0].whole().check_polymer_type(),
        getattr((gemmi.SupSelect),ref_atoms ), )
        
        (flatten_pdb(self.aligned_prot, "chain" )[0].whole()
                                .transform_pos_and_adp(supresult.transform))
        self.transform = supresult.transform
        self.rmsd = supresult.rmsd
        self.rot_matrix = supresult.transform.mat # Rotation Matrix
        self.trans_vect = supresult.transform.vec # Translation Vector
        self.aligned = True

        return self
    
    def calc_rmsd( self, ref_atoms: str = "All") -> float:
        """ 
        Calculate RMSD between ref_prot and mobile_prot without alignment.
        ref_atoms: str
            Atom selection for reference structure alignment. 
            Options: "All", "MainChain", "CaP"
        Returns:
        rmsd: float
            The calculated RMSD value.
        """
        if ref_atoms not in ["All", "MainChain", "CaP"]:
            raise ValueError("ref_atoms must be one of 'All', 'MainChain', or 'CaP'.")

        supresult = gemmi.calculate_current_rmsd( 
        flatten_pdb(self.ref_prot, "chain")[0].whole(),
        flatten_pdb(self.mobile_prot, "chain" )[0].whole(),
        flatten_pdb(self.ref_prot, "chain")[0].whole().check_polymer_type(),
        getattr((gemmi.SupSelect),ref_atoms ), )
        self.rmsd = supresult.rmsd

        return self
    

In [ ]:
align = structAlign( ref_prot, mobile_prot ).align()
ref_prot = align.ref_prot
mobile_prot = align.mobile_prot
aligned_prot = align.aligned_prot

print( "Ref Prot Atom Pos: ", ref_prot[0]["B"][0][0].pos )
print( "Test Prot Atom Pos: ", mobile_prot[0]["A"][0][0].pos )
print( "Aligned Test Prot Atom Pos: ", aligned_prot[0]["A"][0][0].pos )

Ref Prot Atom Pos:  <gemmi.Position(17.43, 6.362, 25.238)>
Test Prot Atom Pos:  <gemmi.Position(20.808, 13.206, 25.917)>
Aligned Test Prot Atom Pos:  <gemmi.Position(17.3858, 6.17196, 25.2535)>


- ### Calc RMSD of prot without alignment

In [7]:
align = structAlign( ref_prot, mobile_prot ).align()
ref_prot = align.ref_prot
mobile_prot = align.mobile_prot
aligned_prot = align.aligned_prot
aligned_rmsd = align.rmsd

# print( "Ref Prot Atom Pos: ", ref_prot[0]["B"][0][0].pos )
# print( "Test Prot Atom Pos: ", mobile_prot[0]["A"][0][0].pos )
# print( "Aligned Test Prot Atom Pos: ", aligned_prot[0]["A"][0][0].pos )

rmsd1= structAlign( ref_prot, mobile_prot ).calc_rmsd().rmsd
rmsd2= structAlign( ref_prot, aligned_prot ).calc_rmsd().rmsd
print( "RMSD before alignment            : ", rmsd1 )
print( "RMSD after alignment             : ", rmsd2 )
print( "RMSD after alignment Ground Truth: ", aligned_rmsd )



RMSD before alignment            :  10.5925665200077
RMSD after alignment             :  0.7513513133630538
RMSD after alignment Ground Truth:  0.75135131336274
